In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

# Step 1: Load and Understand Data
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn (1).csv')
print("Dataset Shape:", df.shape)
print("\n" + "="*50)
print("Column Names and Types:")
print(df.info())
print("\n" + "="*50)
print("First 5 rows:")
print(df.head())
print("\n" + "="*50)
print("Dataset Statistics:")
print(df.describe())
print("\n" + "="*50)
print("Churn Distribution:")
print(df['Churn'].value_counts())

In [ ]:
# Step 2: Data Preprocessing
print("STEP 2: DATA PREPROCESSING")
print("="*50)

# Check for duplicates
print(f"Duplicate rows: {df.duplicated().sum()}")

# Check for missing values
print(f"\nMissing values:\n{df.isnull().sum()}")

# Drop unnecessary columns
df = df.drop(['customerID'], axis=1)
print(f"\nDropped 'customerID' (not useful for prediction)")

# Check TotalCharges column - might be string
print(f"\nTotalCharges dtype: {df['TotalCharges'].dtype}")
# Convert to numeric, coerce errors to NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print(f"After conversion: {df['TotalCharges'].dtype}")

# Handle any NaN from conversion
print(f"Missing TotalCharges after conversion: {df['TotalCharges'].isnull().sum()}")
if df['TotalCharges'].isnull().sum() > 0:
    df = df.dropna(subset=['TotalCharges'])
    print(f"Dropped rows with invalid TotalCharges. New shape: {df.shape}")

print(f"\nFinal dataset shape: {df.shape}")
print(f"Data types:\n{df.dtypes}")

In [ ]:
# Step 3: Exploratory Data Analysis
print("STEP 3: EXPLORATORY DATA ANALYSIS")
print("="*50)

# 1. Churn distribution
plt.figure(figsize=(8, 5))
churn_counts = df['Churn'].value_counts()
plt.bar(churn_counts.index, churn_counts.values, color=['green', 'red'])
plt.title('Churn Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Churn')
plt.ylabel('Count')
plt.show()

# 2. Churn vs Contract type
plt.figure(figsize=(8, 5))
contract_churn = pd.crosstab(df['Contract'], df['Churn'])
contract_churn.plot(kind='bar', color=['green', 'red'])
plt.title('Churn by Contract Type', fontsize=14, fontweight='bold')
plt.xlabel('Contract')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.legend(title='Churn')
plt.tight_layout()
plt.show()

# 3. Churn vs Tenure
plt.figure(figsize=(8, 5))
df.boxplot(column='tenure', by='Churn')
plt.title('Tenure vs Churn', fontsize=14, fontweight='bold')
plt.suptitle('')
plt.xlabel('Churn')
plt.ylabel('Tenure (months)')
plt.show()

# 4. Churn vs InternetService
plt.figure(figsize=(8, 5))
internet_churn = pd.crosstab(df['InternetService'], df['Churn'])
internet_churn.plot(kind='bar', color=['green', 'red'])
plt.title('Churn by Internet Service Type', fontsize=14, fontweight='bold')
plt.xlabel('Internet Service')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.legend(title='Churn')
plt.tight_layout()
plt.show()

# 5. Summary statistics
print("\nChurn Percentage:")
print(df['Churn'].value_counts(normalize=True) * 100)

In [ ]:
# Step 4: Feature Engineering and Encoding
print("STEP 4: FEATURE ENGINEERING & ENCODING")
print("="*50)

# Separate features and target
X = df.drop('Churn', axis=1)
y = df['Churn']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

# Identify categorical and numerical columns
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f"\nCategorical columns: {categorical_cols}")
print(f"Numerical columns: {numerical_cols}")

# Encode categorical variables using LabelEncoder
from sklearn.preprocessing import LabelEncoder

le_dict = {}
for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    le_dict[col] = le
    print(f"Encoded {col}")

# Encode target variable
le_target = LabelEncoder()
y = le_target.fit_transform(y)
print(f"\nTarget encoded: No=0, Yes=1")

print(f"\nFinal features shape: {X.shape}")
print(f"Final target shape: {y.shape}")
print(f"\nFirst few rows of encoded features:")
print(X.head())
print(f"\nTarget distribution: {dict(zip(le_target.classes_, [sum(y==i) for i in range(len(le_target.classes_))]))}")

In [ ]:
# Step 5: Train/Test Split
print("STEP 5: TRAIN/TEST SPLIT")
print("="*50)

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Testing set size: {X_test.shape[0]} samples")
print(f"Total: {X_train.shape[0] + X_test.shape[0]} samples")
print(f"\nTraining features shape: {X_train.shape}")
print(f"Testing features shape: {X_test.shape}")
print(f"\nTraining target distribution:")
print(f"  No (0): {sum(y_train == 0)}")
print(f"  Yes (1): {sum(y_train == 1)}")
print(f"\nTesting target distribution:")
print(f"  No (0): {sum(y_test == 0)}")
print(f"  Yes (1): {sum(y_test == 1)}")

In [ ]:
# Step 6: Train Classification Models
print("STEP 6: TRAINING MODELS")
print("="*50)

# Model 1: Logistic Regression
print("\nTraining Logistic Regression...")
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)
print("Logistic Regression trained!")

# Model 2: Random Forest
print("Training Random Forest...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
print("Random Forest trained!")

# Model 3: Decision Tree (bonus)
print("Training Decision Tree...")
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)
y_pred_dt = dt_model.predict(X_test)
print("Decision Tree trained!")

print("\n" + "="*50)
print("All models trained successfully!")

In [ ]:
# Step 7: Model Evaluation
print("STEP 7: MODEL EVALUATION")
print("="*50)

# Function to evaluate model
def evaluate_model(y_true, y_pred, model_name):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)
    
    print(f"\n{model_name}")
    print("-" * 50)
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print(f"\nConfusion Matrix:")
    print(cm)
    print(f"\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=['No Churn', 'Churn']))
    
    return {'Accuracy': accuracy, 'Precision': precision, 'Recall': recall, 'F1': f1}

# Evaluate all models
results_lr = evaluate_model(y_test, y_pred_lr, "Logistic Regression")
results_rf = evaluate_model(y_test, y_pred_rf, "Random Forest")
results_dt = evaluate_model(y_test, y_pred_dt, "Decision Tree")

# Compare models
print("\n" + "="*50)
print("MODEL COMPARISON")
print("="*50)
comparison = pd.DataFrame({
    'Logistic Regression': results_lr,
    'Random Forest': results_rf,
    'Decision Tree': results_dt
})
print(comparison)

# Find best model
best_model_name = comparison.loc['F1'].idxmax()
print(f"\nBest Model (by F1-Score): {best_model_name}")

In [ ]:
# Step 7B: Improved Model with Class Weights
print("STEP 7B: IMPROVED RANDOM FOREST WITH CLASS WEIGHTS")
print("="*50)

# Retrain Random Forest with balanced class weights
rf_model_improved = RandomForestClassifier(n_estimators=200, max_depth=15, min_samples_split=10, class_weight='balanced', random_state=42)
rf_model_improved.fit(X_train, y_train)
y_pred_improved = rf_model_improved.predict(X_test)

print("Improved Random Forest trained!")

# Evaluate improved model
results_improved = evaluate_model(y_test, y_pred_improved, "Improved Random Forest (Balanced)")

# Compare all models including improved one
print("\n" + "="*50)
print("FINAL MODEL COMPARISON")
print("="*50)
comparison_final = pd.DataFrame({
    'Logistic Regression': results_lr,
    'Random Forest': results_rf,
    'Decision Tree': results_dt,
    'Improved Random Forest': results_improved
})
print(comparison_final)

best_final = comparison_final.loc['F1'].idxmax()
print(f"\nBest Model (by F1-Score): {best_final}")

In [15]:
# Best Accuracy Improvement: GridSearchCV
print("FINDING BEST HYPERPARAMETERS WITH GRIDSEARCHCV")
print("="*50)

from sklearn.model_selection import GridSearchCV

# Define parameter grid
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# GridSearchCV finds best combination
print("Searching for best hyperparameters... (this takes ~1-2 minutes)")
rf_grid = RandomForestClassifier(class_weight='balanced', random_state=42)
grid_search = GridSearchCV(rf_grid, param_grid, cv=5, scoring='f1', n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)

print(f"\nBest Parameters: {grid_search.best_params_}")
print(f"Best CV F1-Score: {grid_search.best_score_:.4f}")

# Use best model
rf_best = grid_search.best_estimator_
y_pred_best = rf_best.predict(X_test)
results_best = evaluate_model(y_test, y_pred_best, "Best Random Forest (GridSearchCV)")

FINDING BEST HYPERPARAMETERS WITH GRIDSEARCHCV
Searching for best hyperparameters... (this takes ~1-2 minutes)
Fitting 5 folds for each of 108 candidates, totalling 540 fits

Best Parameters: {'max_depth': 10, 'min_samples_leaf': 2, 'min_samples_split': 10, 'n_estimators': 100}
Best CV F1-Score: 0.6373

Best Random Forest (GridSearchCV)
--------------------------------------------------
Accuracy:  0.7363
Precision: 0.5028
Recall:    0.7326
F1-Score:  0.5963

Confusion Matrix:
[[762 271]
 [100 274]]

Classification Report:
              precision    recall  f1-score   support

    No Churn       0.88      0.74      0.80      1033
       Churn       0.50      0.73      0.60       374

    accuracy                           0.74      1407
   macro avg       0.69      0.74      0.70      1407
weighted avg       0.78      0.74      0.75      1407



In [16]:
# Step 8: Application - Predict on Sample Customer Data (Using Improved Model)
print("STEP 8: SAMPLE PREDICTIONS")
print("="*50)

# Create sample customer data
sample_customer = pd.DataFrame({
    'SeniorCitizen': [0],
    'tenure': [12],
    'MonthlyCharges': [65.50],
    'TotalCharges': [789.00],
    'gender': ['Male'],
    'Partner': ['Yes'],
    'Dependents': ['No'],
    'PhoneService': ['Yes'],
    'MultipleLines': ['Yes'],
    'InternetService': ['Fiber optic'],
    'OnlineSecurity': ['No'],
    'OnlineBackup': ['No'],
    'DeviceProtection': ['No'],
    'TechSupport': ['No'],
    'StreamingTV': ['Yes'],
    'StreamingMovies': ['No'],
    'Contract': ['Month-to-month'],
    'PaperlessBilling': ['Yes'],
    'PaymentMethod': ['Electronic check']
})

print("Sample Customer Data:")
print(sample_customer)

# Encode the sample data
sample_encoded = sample_customer.copy()
for col in categorical_cols:
    sample_encoded[col] = le_dict[col].transform(sample_encoded[col])

# Reorder columns to match X_train
sample_encoded = sample_encoded[X_train.columns]

print("\nEncoded Sample Data:")
print(sample_encoded)

# Use best model from GridSearchCV
prediction = rf_best.predict(sample_encoded)[0]
prediction_prob = rf_best.predict_proba(sample_encoded)[0]

print("\n" + "="*50)
print("PREDICTION RESULT (Using Improved Model)")
print("="*50)
print(f"Prediction: {'CHURN (Yes)' if prediction == 1 else 'NO CHURN (No)'}")
print(f"Confidence - No Churn: {prediction_prob[0]:.2%}")
print(f"Confidence - Churn: {prediction_prob[1]:.2%}")

if prediction == 1:
    print("\n⚠️ WARNING: This customer is likely to churn!")
    print("Recommended: Offer retention incentives, tech support upgrade, or better contract terms")
else:
    print("\n✓ This customer is likely to stay.")

STEP 8: SAMPLE PREDICTIONS
Sample Customer Data:
   SeniorCitizen  tenure  MonthlyCharges  TotalCharges gender Partner  \
0              0      12            65.5         789.0   Male     Yes   

  Dependents PhoneService MultipleLines InternetService OnlineSecurity  \
0         No          Yes           Yes     Fiber optic             No   

  OnlineBackup DeviceProtection TechSupport StreamingTV StreamingMovies  \
0           No               No          No         Yes              No   

         Contract PaperlessBilling     PaymentMethod  
0  Month-to-month              Yes  Electronic check  

Encoded Sample Data:
   gender  SeniorCitizen  Partner  Dependents  tenure  PhoneService  \
0       1              0        1           0      12             1   

   MultipleLines  InternetService  OnlineSecurity  OnlineBackup  \
0              2                1               0             0   

   DeviceProtection  TechSupport  StreamingTV  StreamingMovies  Contract  \
0                 